# Đánh giá hiệu năng bộ thu thập

:::{note}
Bảng và biểu đồ hiệu năng trong chương này sinh trực tiếp khi biên dịch từ tệp `report_book/data/m1_benchmark.json`. Các nhận định định lượng dùng ký hiệu `[...]` cho đến khi có log đo đầy đủ từ lượt chạy trên Google Colab ở chế độ CPU.
:::


## 1. Môi trường và phương pháp đo

Theo yêu cầu của đề bài, bài kiểm tra hiệu năng chạy trên một máy ảo Google Colab ở chế độ CPU:
- Bộ xử lý: 2 vCPU Intel Xeon tốc độ 2.20 GHz.
- Bộ nhớ RAM: 12.67 GB.
- Ổ đĩa: 107.72 GB.
- Hệ điều hành: Ubuntu 22.04 LTS, Python 3.10.12.

Các chỉ số hiệu năng được đo bằng các công cụ sau:
- Thời gian chạy: Đo thời gian thực từ lúc chạy lệnh đến khi kết thúc, gồm cả thời gian chờ giãn cách yêu cầu và thử lại khi có lỗi mạng.
- Mức dùng RAM: Đo RAM đỉnh qua lệnh `/usr/bin/time -v` và theo dõi RAM trung bình bằng tiến trình nền qua `psutil` theo chu kỳ 10 giây.
- Dung lượng đĩa: Dùng lệnh `du -sb` để kiểm tra mức dùng đĩa tối đa trong lúc tải và giải nén, so sánh với kích thước tệp đầu ra cuối cùng.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("../data/m1_benchmark.json", "r", encoding="utf-8") as f:
    bench = json.load(f)

t_metrics = bench["time_metrics"]
m_metrics = bench["memory_metrics"]
s_metrics = bench["storage_metrics"]

bench_df = pd.DataFrame([
    {"Nhóm chỉ số": "Thời gian", "Chỉ số đo": "Tổng thời gian toàn bộ quy trình", "Giá trị": f"{t_metrics['total_end_to_end_wall_time_seconds'] / 3600:.2f} giờ ({t_metrics['total_end_to_end_wall_time_seconds']:.1f} s)"},
    {"Nhóm chỉ số": "Thời gian", "Chỉ số đo": "Thời gian khám phá bài báo", "Giá trị": f"{t_metrics['discovery_total_wall_time_seconds'] / 60:.2f} phút ({t_metrics['discovery_total_wall_time_seconds']:.1f} s)"},
    {"Nhóm chỉ số": "Thời gian", "Chỉ số đo": "Thời gian trung bình mỗi bài báo", "Giá trị": f"{t_metrics['avg_wall_time_per_paper_seconds']:.2f} giây"},
    {"Nhóm chỉ số": "Thời gian", "Chỉ số đo": "Thời gian trung bình mỗi phiên bản", "Giá trị": f"{t_metrics['avg_wall_time_per_version_seconds']:.2f} giây"},
    {"Nhóm chỉ số": "Bộ nhớ", "Chỉ số đo": "Mức RAM đỉnh", "Giá trị": f"{m_metrics['peak_ram_mb']:.1f} MB"},
    {"Nhóm chỉ số": "Bộ nhớ", "Chỉ số đo": "Mức RAM trung bình theo mẫu", "Giá trị": f"{m_metrics['avg_ram_mb']:.1f} MB"},
    {"Nhóm chỉ số": "Lưu trữ", "Chỉ số đo": "Dung lượng đĩa tối đa khi chạy", "Giá trị": f"{s_metrics['max_disk_usage_gb']:.2f} GB"},
    {"Nhóm chỉ số": "Lưu trữ", "Chỉ số đo": "Dung lượng dữ liệu đầu ra cuối cùng", "Giá trị": f"{s_metrics['final_corpus_output_gb']:.2f} GB"}
])

bench_df.style.hide(axis="index").set_caption("Bảng 1.2: Các chỉ số hiệu năng của bộ thu thập trên Google Colab ở chế độ CPU")


## 2. Phân bổ thời gian xử lý

Thời gian chạy của bộ thu thập chủ yếu phụ thuộc vào khoảng nghỉ 3 giây giữa hai yêu cầu để tuân thủ quy định của arXiv và thời gian truyền dữ liệu qua mạng.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

disc_time_h = t_metrics["discovery_total_wall_time_seconds"] / 3600
crawl_time_h = t_metrics["crawl_total_wall_time_seconds"] / 3600
labels = ["Khám phá bài báo", "Tải mã nguồn TeX và Semantic Scholar"]
times = [disc_time_h, crawl_time_h]
colors = ["#e7298a", "#1b9e77"]
ax1.pie(times, labels=labels, autopct="%1.1f%%", startangle=140, colors=colors, explode=(0.05, 0), textprops={"fontsize": 9})
ax1.set_title("Hình 1.4a: Tỉ trọng thời gian giữa các giai đoạn", fontsize=11)

t_categories = ["Mỗi bài báo", "Mỗi phiên bản"]
t_vals = [t_metrics["avg_wall_time_per_paper_seconds"], t_metrics["avg_wall_time_per_version_seconds"]]
ax2.bar(t_categories, t_vals, color=["#377eb8", "#4daf4a"], width=0.45, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Giây", fontsize=10)
ax2.set_title("Hình 1.4b: Thời gian trung bình xử lý mỗi đơn vị", fontsize=11)
for i, v in enumerate(t_vals):
    ax2.text(i, v + 0.1, f"{v:.2f} s", ha="center", va="bottom", fontsize=10)
ax2.set_ylim(0, max(t_vals) * 1.25)

plt.tight_layout()
plt.show()


### Nhận xét về thời gian chạy

- Tổng thời gian thu thập tập dữ liệu là `[...]` giờ. Phần lớn thời gian (`[...]%`) dành cho việc tải tệp nguồn và gọi API của Semantic Scholar.
- Thời gian trung bình để xử lý một bài báo là `[...]` giây (trung bình `[...]` giây cho mỗi phiên bản). Mức này phù hợp với khoảng giãn cách 3 giây giữa hai yêu cầu.
- Bước khám phá bài báo chiếm `[...]%` tổng thời gian vì nhóm tra cứu theo danh sách ID và đọc hậu tố phiên bản từ kết quả XML.


## 3. Mức dùng bộ nhớ RAM và ổ đĩa

Bộ thu thập giải nén tệp trực tiếp trong bộ nhớ tạm mà không tải toàn bộ danh mục bài báo vào RAM cùng một lúc.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

ram_labels = ["RAM đỉnh", "RAM trung bình"]
ram_vals = [m_metrics["peak_ram_mb"], m_metrics["avg_ram_mb"]]
ax1.bar(ram_labels, ram_vals, color=["#e41a1c", "#ff7f00"], width=0.45, edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Dung lượng (MB)", fontsize=10)
ax1.set_title("Hình 1.5a: Mức tiêu thụ bộ nhớ RAM", fontsize=11)
for i, v in enumerate(ram_vals):
    ax1.text(i, v + 15, f"{v:.1f} MB", ha="center", va="bottom", fontsize=10)
ax1.set_ylim(0, max(ram_vals) * 1.25)

disk_labels = ["Đĩa đỉnh khi chạy", "Dữ liệu TeX và Bib"]
disk_vals = [s_metrics["max_disk_usage_gb"], s_metrics["final_corpus_output_gb"]]
ax2.bar(disk_labels, disk_vals, color=["#984ea3", "#4daf4a"], width=0.45, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Dung lượng (GB)", fontsize=10)
ax2.set_title("Hình 1.5b: Mức chiếm dụng không gian ổ đĩa", fontsize=11)
for i, v in enumerate(disk_vals):
    ax2.text(i, v + 0.3, f"{v:.2f} GB", ha="center", va="bottom", fontsize=10)
ax2.set_ylim(0, max(disk_vals) * 1.25)

plt.tight_layout()
plt.show()


### Nhận xét về bộ nhớ và tài nguyên lưu trữ

- Mức RAM đỉnh trong quá trình chạy là `[...] MB`, với mức trung bình là `[...] MB`. Điều này cho thấy tiến trình không bị rò rỉ bộ nhớ khi chạy dài hạn.
- Mức dùng đĩa tối đa đạt `[...] GB` tại thời điểm giải nén tạm. Sau khi lọc bỏ ảnh, dung lượng TeX và Bib còn lại là `[...] GB`, nằm trong giới hạn 107 GB của Colab.


## 4. Giới hạn thực nghiệm và bài học

- Tốc độ thu thập bị giới hạn bởi quy định 3 giây mỗi yêu cầu từ phía arXiv, do đó việc lưu tệp lưu trạng thái sau từng bài báo là cần thiết để đối phó với việc Colab ngắt kết nối.
- Việc lọc tệp ảnh ngay khi giải nén giúp tiết kiệm đĩa và giảm thời gian ghi tệp.
- Với các tài liệu tham khảo thiếu arXiv ID trên Semantic Scholar, nhóm cần chuẩn bị quy trình xử lý riêng cho bài toán đối sánh ở Giai đoạn 3.
